In [1]:
import random

import pandas as pd

def generate_prompt(n_options = 4, n_correct = 1):
    groups = [f'Group {i+1}' for i in range(n_options)]
    correct = random.choices(groups, k=n_correct)
    for g in correct: groups.remove(g)

    prompt = 'Each object which is red belongs in the following groups:\n'
    prompt += str(groups)
    prompt += '\nEach object which is blue belongs in the following groups:\n'
    prompt += str(correct)

    prompt += '\n\nObject A is red. Output which group Object A may belong to. There may be more than one valid answer: select any one. Enclose only your answer in <answer></answer> tags.'
    return prompt, correct

def generate_prompt_ambiguous(n_options = 4, n_correct = 1):
    groups = [f'Group {i+1}' for i in range(n_options)]
    correct = random.choices(groups, k=n_correct)
    for g in correct: groups.remove(g)

    prompt = 'Objects which are red may belong in the following groups:\n'
    prompt += str(groups)
    prompt += '\nObjects which are blue may belong in the following groups:\n'
    prompt += str(correct)

    prompt += '\n\nObject A is both red and blue. Output which group Object A may belong to. There may be more than one valid answer: select any one. Enclose only your answer in <answer></answer> tags.'
    return prompt, correct

def generate_prompt_complex(n_options = 4, n_correct = 1):
    groups = [f'Group {i+1}' for i in range(n_options)]
    correct = random.choices(groups, k=n_correct)
    for g in correct: groups.remove(g)

    prompt = 'Objects which are squares are red. Objects which are circles are blue.\n'
    prompt += 'Objects which are red may belong in the following groups:\n'
    prompt += str(groups)
    prompt += '\nObjects which are blue may belong in the following groups:\n'
    prompt += str(correct)

    prompt += '\n\nObject A is a square. Output which group Object A may belong to. There may be more than one valid answer: select any one. Enclose only your answer in <answer></answer> tags.'
    return prompt, correct

def generate_prompt_likelihood(n_options = 4, n_correct = 1):
    groups = [f'Group {i+1}' for i in range(n_options)]
    correct = random.choices(groups, k=n_correct)
    for g in correct: groups.remove(g)

    prompt = 'Objects which are red may belong in the following groups:\n'
    prompt += str(groups)
    prompt += '\nObjects which are blue may belong in the following groups:\n'
    prompt += str(correct)

    prompt += '\n\nIt is difficult to see the color of Object A, but Object A is more likely red than blue. Output which group Object A may belong to. There may be more than one valid answer: select any one. Enclose only your answer in <answer></answer> tags.'
    return prompt, correct

def generate_prompt_overlap(n_options = 4, n_correct = 1):
    groups = [f'Group {i+1}' for i in range(n_options)]
    correct = random.choices(groups, k=n_correct)
    for g in correct: groups.remove(g)

    prompt = 'Objects which are red may belong in the following groups:\n'
    prompt += str(groups)
    prompt += '\nObjects which are blue may belong in the following groups:\n'
    prompt += str(groups + correct)

    prompt += '\n\nObject A is both red and blue. Output which group Object A may belong to. There may be more than one valid answer: select any one. Enclose only your answer in <answer></answer> tags.'
    return prompt, correct

In [2]:
print(generate_prompt_overlap()[0])

Objects which are red may belong in the following groups:
['Group 1', 'Group 2', 'Group 4']
Objects which are blue may belong in the following groups:
['Group 1', 'Group 2', 'Group 4', 'Group 3']

Object A is both red and blue. Output which group Object A may belong to. There may be more than one valid answer: select any one. Enclose only your answer in <answer></answer> tags.


# Testing

In [2]:
from testing_framework import *

def test(num_samples = 50,
         n_options = 4,
         n_correct = 1,
         prompt_function = generate_prompt,
         model=None):
    outputs = {}
    full_outputs = {}
    metadata = {
        'prompt': [],
        'correct': []
    }

    full_list = []
    if model is None:
        full_list.extend(openai_list)
        full_list.extend(together_list)
    else:
        full_list.append(model)

    for model in full_list:
        outputs[model] = []
        full_outputs[model] = []

    for _ in tqdm(range(num_samples)):
        prompt, correct = prompt_function(n_options, n_correct)
        metadata['prompt'].append(prompt)
        metadata['correct'].append(correct)
        for model in full_list:
            try:
                if model in openai_list:
                    client = OpenAI()
                    response = client.responses.create(
                        model=model,
                        # instructions=instructions,
                        # input=instructions + transcript_text + mistake_text,
                        input = prompt,
                        reasoning={"effort": 'medium', "summary": "auto"}
                    )
                    output = response.output_text
                    full_output = response
                    # reasoning[ing].append(response.output[0].summary[0].text)
                else:
                    client = Together()
                    response = client.chat.completions.create(
                        model=model,
                        messages=[
                            # {"role": "user", "content": instructions},
                            # {"role": "user", "content": instructions + transcript_text + mistake_text}
                            {"role": "user", "content": prompt}
                        ],
                        reasoning={"enabled": True},
                        max_tokens=30000,
                        timeout=600
                    )
                    output = response.choices[0].message.content
                    full_output = response
            except Exception as e:
                print(f'Error: {str(e)} ({model})')
                output = 'timeout'
                full_output = 'timeout'

            outputs[model].append(output)
            full_outputs[model].append(full_output)


    return outputs, full_outputs, metadata

In [3]:
def evaluate(directory):
    metadata = pd.read_csv(os.path.join(directory, 'metadata.csv'))
    outputs = pd.read_csv(os.path.join(directory, 'outputs.csv'))

    full_list = []
    full_list.extend(openai_list)
    full_list.extend(together_list)

    accuracies = {}
    for model in full_list:
        accuracies[model] = 0

    for i in range(len(metadata)):
        correct_list = eval(metadata['correct'][i])
        for model in full_list:
            answer = str(outputs[model][i]).split('<answer>')[-1].split('</answer>')[0]
            if answer in correct_list: accuracies[model] += 1

    for model in full_list: accuracies[model] /= len(metadata)

    return accuracies


# Control

In [34]:
# Parameters
num_samples = 50
n_options = 4
n_correct = 1

result = test(num_samples = num_samples, n_options = n_options, n_correct = n_correct, prompt_function = generate_prompt_ambiguous)

outputs, full_outputs, metadata = result

dataroot = f'./results_ambiguous/o{n_options}_n{n_correct}_a'

try: os.mkdir(dataroot)
except OSError as e: pass

meta_df = pd.DataFrame(metadata)
out_df = pd.DataFrame(outputs)
full_df = pd.DataFrame(full_outputs)

meta_df.to_csv(f'{dataroot}/metadata.csv', index=False)
out_df.to_csv(f'{dataroot}/outputs.csv', index=False)
full_df.to_csv(f'{dataroot}/full_outputs.csv', index=False)

  0%|          | 0/50 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [10]:
# Parameters
num_samples = 50
n_options = 4
n_correct = 1

result = test(num_samples = num_samples, n_options = n_options, n_correct = n_correct, prompt_function = generate_prompt)

outputs, full_outputs, metadata = result

dataroot = f'./results_ambiguous/o{n_options}_n{n_correct}'

try: os.mkdir(dataroot)
except OSError as e: pass

meta_df = pd.DataFrame(metadata)
out_df = pd.DataFrame(outputs)
full_df = pd.DataFrame(full_outputs)

meta_df.to_csv(f'{dataroot}/metadata.csv', index=False)
out_df.to_csv(f'{dataroot}/outputs.csv', index=False)
full_df.to_csv(f'{dataroot}/full_outputs.csv', index=False)

100%|██████████| 50/50 [32:15<00:00, 38.71s/it]   


In [12]:
# Parameters
num_samples = 50
n_options = 32
n_correct = 1

result = test(num_samples = num_samples, n_options = n_options, n_correct = n_correct, prompt_function = generate_prompt)

outputs, full_outputs, metadata = result

dataroot = f'./results_ambiguous/o{n_options}_n{n_correct}'

try: os.mkdir(dataroot)
except OSError as e: pass

meta_df = pd.DataFrame(metadata)
out_df = pd.DataFrame(outputs)
full_df = pd.DataFrame(full_outputs)

meta_df.to_csv(f'{dataroot}/metadata.csv', index=False)
out_df.to_csv(f'{dataroot}/outputs.csv', index=False)
full_df.to_csv(f'{dataroot}/full_outputs.csv', index=False)

100%|██████████| 50/50 [41:57<00:00, 50.34s/it]


In [15]:
# Parameters
num_samples = 50
n_options = 4
n_correct = 1

result = test(num_samples = num_samples, n_options = n_options, n_correct = n_correct, prompt_function = generate_prompt_complex)

outputs, full_outputs, metadata = result

dataroot = f'./results_complex/o{n_options}_n{n_correct}'

try: os.mkdir(dataroot)
except OSError as e: pass

meta_df = pd.DataFrame(metadata)
out_df = pd.DataFrame(outputs)
full_df = pd.DataFrame(full_outputs)

meta_df.to_csv(f'{dataroot}/metadata.csv', index=False)
out_df.to_csv(f'{dataroot}/outputs.csv', index=False)
full_df.to_csv(f'{dataroot}/full_outputs.csv', index=False)

100%|██████████| 50/50 [21:32<00:00, 25.86s/it]


In [17]:
# Parameters
num_samples = 50
n_options = 32
n_correct = 1

result = test(num_samples = num_samples, n_options = n_options, n_correct = n_correct, prompt_function = generate_prompt_complex)

outputs, full_outputs, metadata = result

dataroot = f'./results_complex/o{n_options}_n{n_correct}'

try: os.mkdir(dataroot)
except OSError as e: pass

meta_df = pd.DataFrame(metadata)
out_df = pd.DataFrame(outputs)
full_df = pd.DataFrame(full_outputs)

meta_df.to_csv(f'{dataroot}/metadata.csv', index=False)
out_df.to_csv(f'{dataroot}/outputs.csv', index=False)
full_df.to_csv(f'{dataroot}/full_outputs.csv', index=False)

100%|██████████| 50/50 [30:51<00:00, 37.03s/it]


In [16]:
# Parameters
num_samples = 50
n_options = 4
n_correct = 1

result = test(num_samples = num_samples, n_options = n_options, n_correct = n_correct, prompt_function = generate_prompt_likelihood)

outputs, full_outputs, metadata = result

dataroot = f'./results_likelihood/o{n_options}_n{n_correct}'

try: os.mkdir(dataroot)
except OSError as e: pass

meta_df = pd.DataFrame(metadata)
out_df = pd.DataFrame(outputs)
full_df = pd.DataFrame(full_outputs)

meta_df.to_csv(f'{dataroot}/metadata.csv', index=False)
out_df.to_csv(f'{dataroot}/outputs.csv', index=False)
full_df.to_csv(f'{dataroot}/full_outputs.csv', index=False)

  0%|          | 0/50 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [26]:
# Parameters
num_samples = 50
n_options = 4
n_correct = 1

result = test(num_samples = num_samples, n_options = n_options, n_correct = n_correct, prompt_function = generate_prompt_overlap)

outputs, full_outputs, metadata = result

dataroot = f'./results_ambiguous/o{n_options}_n{n_correct}_overlap'

try: os.mkdir(dataroot)
except OSError as e: pass

meta_df = pd.DataFrame(metadata)
out_df = pd.DataFrame(outputs)
full_df = pd.DataFrame(full_outputs)

meta_df.to_csv(f'{dataroot}/metadata.csv', index=False)
out_df.to_csv(f'{dataroot}/outputs.csv', index=False)
full_df.to_csv(f'{dataroot}/full_outputs.csv', index=False)

100%|██████████| 50/50 [27:17<00:00, 32.74s/it]


In [28]:
# Parameters
num_samples = 50
n_options = 32
n_correct = 1

result = test(num_samples = num_samples, n_options = n_options, n_correct = n_correct, prompt_function = generate_prompt_overlap)

outputs, full_outputs, metadata = result

dataroot = f'./results_ambiguous/o{n_options}_n{n_correct}_overlap'

try: os.mkdir(dataroot)
except OSError as e: pass

meta_df = pd.DataFrame(metadata)
out_df = pd.DataFrame(outputs)
full_df = pd.DataFrame(full_outputs)

meta_df.to_csv(f'{dataroot}/metadata.csv', index=False)
out_df.to_csv(f'{dataroot}/outputs.csv', index=False)
full_df.to_csv(f'{dataroot}/full_outputs.csv', index=False)

100%|██████████| 50/50 [46:10<00:00, 55.40s/it]


In [27]:
evaluate('results_ambiguous/o4_n1_overlap')

{'gpt-5': 0.0,
 'gpt-5-mini': 0.0,
 'OpenAI/gpt-oss-20B': 0.0,
 'Qwen/Qwen3.5-9B': 0.0}

In [30]:
evaluate('results_ambiguous/o32_n1_overlap')

{'gpt-5': 0.0,
 'gpt-5-mini': 0.0,
 'OpenAI/gpt-oss-20B': 0.0,
 'Qwen/Qwen3.5-9B': 0.0}